# 🤿 DeepClean v4 — Adaptive Dual-Mode Underwater Trash Detection

**End-to-end demo notebook — runs fully in Google Colab (no GPU required for the synthetic demo)**

This notebook walks through:
1. Environment setup
2. Generating a synthetic 300-frame underwater test video
3. Running the 3-Layer Adaptive Controller
4. Visualising Complexity Score trajectory and model-selection timeline
5. Statistical significance tests
6. Layer-combination ablation study
7. All paper figures

---
**Paper:** *Adaptive Dual-Mode Underwater Trash Detection System*  
**Authors:** Amol Bhatia, Krisvarish V., Harsh Yadav, Avishkar Jaiswal, Shivani Gupta, Saurav Gupta

In [ ]:
# ── 1. Clone repo & install dependencies ─────────────────────────────────────
import os

REPO = 'DeepClean-AdaptiveDualMode'
if not os.path.exists(REPO):
    !git clone https://github.com/your-org/{REPO}.git

%cd {REPO}

!pip install -q opencv-python-headless numpy scipy matplotlib scikit-image tqdm rich

In [ ]:
# ── 2. Generate synthetic underwater video ────────────────────────────────────
print('Generating synthetic_underwater.mp4 (300 frames @ 30 FPS)...')
!python generate_synthetic_video.py

# Quick sanity check
import os
size_kb = os.path.getsize('synthetic_underwater.mp4') / 1024
print(f'Video file size: {size_kb:.1f} KB')

In [ ]:
# ── 3. Run the full adaptive pipeline ────────────────────────────────────────
!python run_adaptive_test.py --video synthetic_underwater.mp4 --output results/

import os
print('\nGenerated files:')
for f in sorted(os.listdir('results')):
    print(f'  {f}')

In [ ]:
# ── 4. Display CS trajectory plot ────────────────────────────────────────────
from IPython.display import Image, display

img_path = 'results/cs_trajectory.png'
if os.path.exists(img_path):
    display(Image(img_path, width=900))
else:
    print('Plot not found — re-running generate_plots.py...')
    !python evaluation/generate_plots.py --output results/

In [ ]:
# ── 5. Live CS computation demo ───────────────────────────────────────────────
import sys, cv2, numpy as np
sys.path.insert(0, '.')

from controller.deepclean_v4 import DeepCleanController
from controller.complexity_score import ComplexityScoreComputer
from controller.parameter_extractor import ParameterExtractor

ctrl = DeepCleanController()
ext  = ParameterExtractor()
cap  = cv2.VideoCapture('synthetic_underwater.mp4')

frame_results = []
fi = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    res = ctrl.step(frame)
    frame_results.append({
        'frame': fi,
        'model': res.model_name,
        'cs_smooth': res.cs_smooth,
        'cs_raw': res.cs_raw,
        'switched': res.switched,
        'controller_ms': res.controller_ms,
    })
    fi += 1

cap.release()
print(f'Processed {fi} frames')
print(f'Summary: {ctrl.summary()}')

In [ ]:
# ── 6. Plot CS trajectory inline ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

frames   = [r['frame']     for r in frame_results]
cs       = [r['cs_smooth'] for r in frame_results]
cs_raw   = [r['cs_raw']    for r in frame_results]
models   = [r['model']     for r in frame_results]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# Hysteresis band
ax1.axhspan(0.40, 0.55, alpha=0.12, color='orange', label='Hysteresis zone')
ax1.axhline(0.40, color='orange', lw=1.2, ls='--')
ax1.axhline(0.55, color='orange', lw=1.2, ls='--')

# Phase shading
for x0, x1, col, lbl in [
    (0,   100, '#2ECC71', 'Phase 1: Clear'),
    (100, 200, '#E74C3C', 'Phase 2: Turbid'),
    (200, 300, '#9B59B6', 'Phase 3: Moderate'),
]:
    ax1.axvspan(x0, x1, color=col, alpha=0.06)
    ax1.text((x0+x1)/2, 0.02, lbl, ha='center', fontsize=8, color=col, fontweight='bold')

ax1.plot(frames, cs_raw, color='#BDC3C7', lw=0.8, alpha=0.5, label='CS_raw')
ax1.plot(frames, cs,     color='#3498DB', lw=2.0, label='CS_smooth (EMA α=0.3)')
ax1.set_ylabel('Complexity Score (CS̃)', fontsize=11)
ax1.set_ylim(-0.02, 1.02)
ax1.legend(loc='upper right', fontsize=9)
ax1.set_title('DeepClean v4 — Complexity Score Trajectory & Model Selection', fontsize=12)

# Model strip
colours = ['#2ECC71' if m == 'YOLOv8n' else '#E74C3C' for m in models]
ax2.bar(frames, [1]*len(frames), color=colours, width=1.0, linewidth=0)
ax2.set_yticks([])
ax2.set_ylabel('Model', fontsize=9)
ax2.set_xlabel('Frame Index', fontsize=11)

plt.tight_layout()
plt.savefig('results/cs_trajectory_inline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/cs_trajectory_inline.png')

In [ ]:
# ── 7. Controller overhead timing ─────────────────────────────────────────────
import numpy as np

overheads = [r['controller_ms'] for r in frame_results]
print('Controller Overhead Statistics (ms)')
print(f'  Mean  : {np.mean(overheads):.3f} ms')
print(f'  Std   : {np.std(overheads):.3f} ms')
print(f'  Max   : {np.max(overheads):.3f} ms')
print(f'  P99   : {np.percentile(overheads, 99):.3f} ms')
print(f'\n  Target: < 1.0 ms → {"✓ PASS" if np.mean(overheads) < 1.0 else "✗ FAIL"}')

In [ ]:
# ── 8. Statistical significance tests ────────────────────────────────────────
!python evaluation/run_statistical_tests.py --output results/

In [ ]:
# ── 9. Layer-combination ablation study ──────────────────────────────────────
!python evaluation/run_ablation.py --video synthetic_underwater.mp4 --output results/

In [ ]:
# ── 10. Model benchmark (simulation mode) ─────────────────────────────────────
!python evaluation/benchmark_models.py --simulate --output results/

In [ ]:
# ── 11. Generate all paper figures ────────────────────────────────────────────
!python evaluation/generate_plots.py --output results/figures

from IPython.display import Image, display
import os

for fig_name in sorted(os.listdir('results/figures')):
    if fig_name.endswith('.png'):
        print(f'\n--- {fig_name} ---')
        display(Image(f'results/figures/{fig_name}', width=800))

In [ ]:
# ── 12. Unit tests ────────────────────────────────────────────────────────────
!pip install -q pytest
!python -m pytest tests/ -v --tb=short 2>&1 | head -80

In [ ]:
# ── 13. Objective function evaluation ─────────────────────────────────────────
from controller.objective_function import UtilityEvaluator

ev   = UtilityEvaluator(lam=0.60)
hist = [r['model'] for r in frame_results]

comparison = ev.compare_baselines(hist)
ev.print_report(comparison)

print(f"\nAdaptive J     = {comparison['Adaptive']['J']:.5f}")
print(f"Energy savings = {comparison['Adaptive']['energy_savings_pct']:.1f}%")
print(f"Switch rate    = {comparison['Adaptive']['switch_rate']:.5f} per frame")

## Summary

| Metric | Value |
|---|---|
| Controller overhead | < 1 ms/frame |
| Energy savings vs YOLOv8x | ~69% |
| mAP@0.5 (adaptive) | 0.910 |
| Switch rate | ≤ 4.1 / 1000 frames |
| Statistical significance | p < 0.001 (all comparisons) |
| Cohen's d | ≈ 3.0 (large effect) |

The DeepClean v4 Adaptive Dual-Mode controller successfully balances detection accuracy and energy efficiency for AUV-based underwater trash detection missions.

---
*For training on real datasets or deploying on hardware, see the full repository README.*